In [1]:
%load_ext dotenv
%dotenv

In [6]:
import pandas as pd
import pinecone
from pinecone import Pinecone, ServerlessSpec
import os
from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer
import hashlib
import pdfplumber
from pathlib import Path
import re

In [7]:
pc = Pinecone(api_key = os.environ.get('PINECONE_API_KEY'), environment = os.environ.get('PINECONE_ENV'))

In [8]:
PDF_FOLDER = "CoinCleaning"  
CHUNK_SIZE = 500 
CHUNK_OVERLAP = 50  

In [9]:
def parse_filename_metadata(filepath: str) -> dict:
    """
    Parses metadata from filename formatted as:
        <DocName>_<ShortDescription>_<AuthorName>.pdf
 
    Returns a dict with keys: doc_name, description, author.
    Falls back gracefully if the filename doesn't match the convention.
    """
    stem = Path(filepath).stem          # filename without extension
    parts = stem.split("_", maxsplit=2) # split on first 2 underscores only
 
    if len(parts) == 3:
        doc_name, description, author = parts
    elif len(parts) == 2:
        doc_name, description = parts
        author = "Unknown"
    else:
        doc_name = parts[0]
        description = ""
        author = "Unknown"
 
    # Convert CamelCase or underscored tokens to readable strings
    doc_name_readable    = re.sub(r'(?<!^)(?=[A-Z])', ' ', doc_name).strip()
    description_readable = re.sub(r'(?<!^)(?=[A-Z])', ' ', description).strip()
    author_readable      = re.sub(r'(?<!^)(?=[A-Z])', ' ', author).strip()
 
    return {
        "doc_name":    doc_name_readable,
        "description": description_readable,
        "author":      author_readable,
        "source_file": Path(filepath).name,
    }

In [10]:
def extract_text_from_pdf(filepath: str) -> str:
    """
    Extracts all text from a PDF using pdfplumber.
    Joins pages with double newlines to preserve paragraph boundaries.
    Falls back to pypdf if pdfplumber yields nothing.
    """
    full_text = []
 
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                full_text.append(page_text.strip())
 
    if not full_text:
        # Fallback: try pypdf
        from pypdf import PdfReader
        reader = PdfReader(filepath)
        for page in reader.pages:
            t = page.extract_text()
            if t:
                full_text.append(t.strip())
 
    return "\n\n".join(full_text)

In [11]:
def clean_text(text: str) -> str:
    """
    Light cleaning to fix common PDF extraction artifacts:
    - Collapse excessive whitespace within lines
    - Normalize line endings
    - Remove lines that are purely page numbers or headers/footers (heuristic)
    """
    lines = text.splitlines()
    cleaned = []
 
    for line in lines:
        line = line.strip()
        # Drop very short lines that are likely page numbers or artefacts
        if len(line) <= 3 and re.match(r'^[\d\s\-–—]*$', line):
            continue
        # Collapse internal whitespace
        line = re.sub(r'[ \t]+', ' ', line)
        cleaned.append(line)
 
    return "\n".join(cleaned)

In [12]:
def split_into_paragraphs(text: str) -> list[str]:
    """
    Splits text on blank lines (paragraph boundaries), which pdfplumber
    typically preserves when a document has clear paragraph structure.
    """
    # Split on one or more blank lines
    raw_paragraphs = re.split(r'\n{2,}', text)
    # Filter out empty/whitespace-only segments
    return [p.strip() for p in raw_paragraphs if p.strip()]

In [13]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """
    Chunks text in a paragraph-aware way:
      1. Splits by paragraph boundaries first.
      2. Accumulates paragraphs into chunks up to `chunk_size` words.
      3. Adds `overlap` words of the previous chunk to the start of the next
         for context continuity (important for RAG retrieval quality).
 
    Returns a list of text chunk strings.
    """
    paragraphs = split_into_paragraphs(text)
    chunks = []
    current_words = []
 
    for para in paragraphs:
        para_words = para.split()
 
        # If adding this paragraph would exceed chunk_size, flush current chunk
        if current_words and len(current_words) + len(para_words) > chunk_size:
            chunk_text_str = " ".join(current_words)
            chunks.append(chunk_text_str)
            # Start new chunk with overlap from the end of the previous chunk
            current_words = current_words[-overlap:] if overlap > 0 else []
 
        current_words.extend(para_words)
 
        # If a single paragraph is already larger than chunk_size, force-split it
        while len(current_words) > chunk_size:
            chunks.append(" ".join(current_words[:chunk_size]))
            current_words = current_words[chunk_size - overlap:]
 
    # Flush the final chunk
    if current_words:
        chunks.append(" ".join(current_words))
 
    return chunks

In [14]:
def process_pdf_folder(folder: str) -> list[dict]:
    """
    Processes all PDFs in `folder`.
    Returns a flat list of chunk dicts, each shaped like:
 
    {
        "doc_name":    "Electrochemical Cleaning",
        "description": "Electrolysis Methods For Coins",
        "author":      "John Smith",
        "source_file": "ElectrochemicalCleaning_ElectrolysisMethodsForCoins_JohnSmith.pdf",
        "chunk_index": 3,           # 0-based position within this document
        "total_chunks": 12,         # total chunks from this document
        "text":        "..."        # the actual chunk text for embedding
    }
    """
    all_chunks = []
    pdf_paths = sorted(Path(folder).glob("*.pdf"))
 
    if not pdf_paths:
        print(f"No PDFs found in '{folder}'. Check the folder path.")
        return all_chunks
 
    for pdf_path in pdf_paths:
        print(f"\nProcessing: {pdf_path.name}")
 
        # 1. Parse metadata from filename
        metadata = parse_filename_metadata(str(pdf_path))
        print(f"  → Doc: {metadata['doc_name']} | Author: {metadata['author']}")
 
        # 2. Extract raw text
        raw_text = extract_text_from_pdf(str(pdf_path))
        if not raw_text.strip():
            print(f"  ⚠ No text extracted — PDF may be scanned. Consider OCR.")
            continue
 
        # 3. Clean the text
        cleaned = clean_text(raw_text)
 
        # 4. Chunk the text
        chunks = chunk_text(cleaned, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
        print(f"  → {len(chunks)} chunks produced")
 
        # 5. Build chunk dicts
        for i, chunk in enumerate(chunks):
            record = {
                **metadata,              # doc_name, description, author, source_file
                "chunk_index":  i,
                "total_chunks": len(chunks),
                "text":         chunk,
            }
            all_chunks.append(record)
 
    print(f"\n✓ Total chunks across all documents: {len(all_chunks)}")
    return all_chunks

In [15]:
chunks = process_pdf_folder(PDF_FOLDER)


Processing: AMethodForCleaningAncientBronzeCoins_ExpertCleaner_SaulRoll.pdf
  → Doc: A Method For Cleaning Ancient Bronze Coins | Author: Saul Roll
  → 27 chunks produced

Processing: CopperAndBronzeInArtCorrosionColorantsConservation_MuseumCopperCare_DavidAScott.pdf
  → Doc: Copper And Bronze In Art Corrosion Colorants Conservation | Author: David A Scott
  → 470 chunks produced

Processing: OnTheRemovalOfHornSilverOnAncientSilverCoins_HornSilverTreatment_YdeJong.pdf
  → Doc: On The Removal Of Horn Silver On Ancient Silver Coins | Author: Yde Jong
  → 6 chunks produced

Processing: TheArtOfCleaningAndRestorationOfAncientCoinsAndArtifacts_NobleRomanCoins_KevinRSandes.pdf
  → Doc: The Art Of Cleaning And Restoration Of Ancient Coins And Artifacts | Author: Kevin R Sandes
  → 73 chunks produced

✓ Total chunks across all documents: 576


In [16]:
for record in chunks[:2]:
    print("\n── CHUNK PREVIEW ──────────────────────────────────────────────")
    for key, val in record.items():
        if key == "text":
            print(f"  text (first 300 chars): {val[:300]}...")
        else:
            print(f"  {key}: {val}")


── CHUNK PREVIEW ──────────────────────────────────────────────
  doc_name: A Method For Cleaning Ancient Bronze Coins
  description: Expert Cleaner
  author: Saul Roll
  source_file: AMethodForCleaningAncientBronzeCoins_ExpertCleaner_SaulRoll.pdf
  chunk_index: 0
  total_chunks: 27
  text (first 300 chars): A Method for Cleaning Ancient Bronze Coins With a long introduction, and much annoying detail Plus some commentaries on bronze disease To which is added a Bitter Afterword And a Bitchy Epilogue, followed by an Appendix of coin photographs Saúl Roll www.romanorum.com© BIG SMALL PRINT: Disclaimer: Thi...

── CHUNK PREVIEW ──────────────────────────────────────────────
  doc_name: A Method For Cleaning Ancient Bronze Coins
  description: Expert Cleaner
  author: Saul Roll
  source_file: AMethodForCleaningAncientBronzeCoins_ExpertCleaner_SaulRoll.pdf
  chunk_index: 1
  total_chunks: 27
  text (first 300 chars): guidelines reflect the method that works for me after having spent over tw

In [17]:
pc = Pinecone(api_key = os.environ.get('PINECONE_API_KEY'), environment = os.environ.get('PINECONE_ENV'))

In [19]:
INDEX_NAME = "ancient-coin-cleaning"
EMBEDDING_DIM = 1536  # text-embedding-3-small output dimension
METRIC = "cosine"

# Create index if it doesn't already exist
existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric=METRIC,
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"✓ Index '{INDEX_NAME}' created.")
else:
    print(f"→ Index '{INDEX_NAME}' already exists, skipping creation.")

index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())

✓ Index 'ancient-coin-cleaning' created.
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 26 Mar 2026 18:18:41 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '45',
                                    'x-pinecone-request-latency-ms': '44',
                                    'x-pinecone-response-duration-ms': '46'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [20]:
from openai import OpenAI
import time

openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
EMBEDDING_MODEL = "text-embedding-3-small"
BATCH_SIZE = 100  # OpenAI allows up to 2048 inputs per request; 100 is safe

def embed_chunks(chunks: list[dict], model: str = EMBEDDING_MODEL) -> list[dict]:
    """
    Adds an 'embedding' key to each chunk dict.
    Processes in batches to stay within API rate limits.
    """
    texts = [c["text"] for c in chunks]
    all_embeddings = []

    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i : i + BATCH_SIZE]
        response = openai_client.embeddings.create(input=batch, model=model)
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f"  Embedded {min(i + BATCH_SIZE, len(texts))}/{len(texts)} chunks...")
        time.sleep(0.25)  # gentle rate-limit buffer

    for chunk, embedding in zip(chunks, all_embeddings):
        chunk["embedding"] = embedding

    print(f"✓ All {len(chunks)} chunks embedded.")
    return chunks

chunks = embed_chunks(chunks)


  Embedded 100/576 chunks...
  Embedded 200/576 chunks...
  Embedded 300/576 chunks...
  Embedded 400/576 chunks...
  Embedded 500/576 chunks...
  Embedded 576/576 chunks...
✓ All 576 chunks embedded.


In [21]:
import hashlib

UPSERT_BATCH_SIZE = 100  # Pinecone recommends 100 vectors per upsert batch

def make_vector_id(chunk: dict) -> str:
    """
    Creates a stable, unique ID from source file + chunk index.
    Using a hash avoids issues with special characters in filenames.
    """
    raw = f"{chunk['source_file']}__chunk_{chunk['chunk_index']}"
    return hashlib.md5(raw.encode()).hexdigest()

def upsert_chunks(index, chunks: list[dict], batch_size: int = UPSERT_BATCH_SIZE):
    """
    Upserts all embedded chunks to Pinecone.
    Metadata fields (everything except the raw embedding) are stored
    alongside the vector for filtered retrieval later.
    """
    vectors = []
    for chunk in chunks:
        vector_id = make_vector_id(chunk)
        metadata = {
            "doc_name":     chunk["doc_name"],
            "description":  chunk["description"],
            "author":       chunk["author"],
            "source_file":  chunk["source_file"],
            "chunk_index":  chunk["chunk_index"],
            "total_chunks": chunk["total_chunks"],
            "text":         chunk["text"],   # stored for retrieval — returned with matches
        }
        vectors.append({
            "id":       vector_id,
            "values":   chunk["embedding"],
            "metadata": metadata,
        })

    # Upsert in batches
    for i in range(0, len(vectors), batch_size):
        batch = vectors[i : i + batch_size]
        index.upsert(vectors=batch)
        print(f"  Upserted {min(i + batch_size, len(vectors))}/{len(vectors)} vectors...")

    print(f"✓ All {len(vectors)} vectors upserted to '{INDEX_NAME}'.")

upsert_chunks(index, chunks)

# Confirm final state
print("\nIndex stats after upsert:")
print(index.describe_index_stats())

  Upserted 100/576 vectors...
  Upserted 200/576 vectors...
  Upserted 300/576 vectors...
  Upserted 400/576 vectors...
  Upserted 500/576 vectors...
  Upserted 576/576 vectors...
✓ All 576 vectors upserted to 'ancient-coin-cleaning'.

Index stats after upsert:
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '186',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 26 Mar 2026 18:22:37 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '4',
                                    'x-pinecone-request-latency-ms': '4',
                                    'x-pinecone-response-duration-ms': '6'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vecto